# Two Traders with Private Markov-Chain Signals

The game of `two_traders.ipynb` with each signal replaced by a two-state Markov chain $\alpha^i_t \in \{-1, 1\}$
switching at rate $\kappa = 1$, private to trader $i$. The conditional expectation of the other signal is the
Wonham filter, nonlinear in the observed path, so the linear-strategy solver does not apply.

At $\lambda = 0$ the traders decouple. The exact solution combines the discrete Bayes filter of the other chain with
a backward recursion for each trader, and is the reference. The equilibrium among strategies linear in the observed
path is the baseline.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
import pig
from pig import Solver, relative_l2, use_style, LEARNED, LINEAR
use_style()
device = pig.use_device()
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from pig.games import MarkovTraders, markov_traders_exact_lam0, linear_equilibrium
game = MarkovTraders(lam=0.0)

## 1. Learned, linear and exact equilibria at $\lambda = 0$

- Learned: as in `two_traders.ipynb`, with damping 0.5 and a control variate that also includes the compensated
  chain increments $\alpha^i_{t_{k+1}} - e^{-2\kappa\Delta t}\alpha^i_{t_k}$.
- Linear strategies: the same iteration and targets, with each regression replaced by least squares at each time on
  the current observation and all past observation increments.

In [ ]:
solver = Solver(game, damping=0.5, steps=750, batch=2048, n_paths=16384, control_variate=True)
exact = to_players(markov_traders_exact_lam0(game, solver.validation))
learned, _ = solver.run(10, average_from=4, reference=exact)
linear = linear_equilibrium(solver, iterations=20, average_from=10, log=None)
for label, controls in (('learned', learned), ('linear strategies', linear)):
    print(f'{label:18s} relative L2 error: {relative_l2(controls[0], exact[0]):.3f}, {relative_l2(controls[1], exact[1]):.3f}')

In [ ]:
def plot_controls(learned, exact, titles, linear=None, paths=(0, 1, 2), ylabel='control'):
    t = np.arange(learned[0].shape[1]) * game.dt
    fig, axes = plt.subplots(1, len(learned), figsize=(10, 3.2), sharey=True)
    for i, ax in enumerate(axes):
        for n, path in enumerate(paths):
            ax.plot(t, learned[i][path].cpu(), color=LEARNED, lw=2.6, alpha=0.45, label='learned' if n == 0 else None)
            if linear is not None:
                ax.plot(t, linear[i][path].cpu(), color=LINEAR, lw=1.0, label='linear strategies' if n == 0 else None)
            ax.plot(t, exact[i][path].cpu(), 'k--', lw=0.8, label='exact' if n == 0 else None)
        ax.set_xlabel('$t$'); ax.set_title(titles[i])
    axes[0].set_ylabel(ylabel); axes[0].legend()
    plt.show()

plot_controls(learned, exact, ['trader 1', 'trader 2'], linear=linear, ylabel=r'$\nu_t$')

## 2. Private information at $\lambda = 1$

At $\lambda = 1$ no exact solution is known. The first order condition residual measures how far each control is
from the regression of its own target, fitted afresh on new paths.

In [ ]:
game = MarkovTraders(lam=1.0)
solver = Solver(game, damping=0.6, steps=750, batch=2048, n_paths=16384, control_variate=True)
learned, _ = solver.run(10, average_from=4, log=lambda message: None)
print('first order condition residuals:', [round(pig.first_order_residual(solver, i), 3) for i in range(2)])
print('mean |nu|:', [round(float(c.abs().mean()), 4) for c in learned])